### Phương pháp Lặp Jacobi (Hệ phương trình Ax=b)


In [7]:
import numpy as np
from IPython.display import display, Math, Markdown

def matrix_to_latex(M, precision=4):
    if not isinstance(M, np.ndarray): return str(M)
    elif M.ndim == 1:
        inner = " \\\\ ".join([f"{x:.{precision}f}" for x in M])
        return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"
    else:
        rows = " \\\\ ".join([" & ".join([f"{x:.{precision}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"

def Jacobi_Ax_b(A_input, b_input, x0_input=None, num_iters=None, epsilon=None):
    A = np.array(A_input, dtype=float)
    b = np.array(b_input, dtype=float).flatten()
    n = len(b)
    
    if x0_input is None: x_k = np.zeros(n)
    else: x_k = np.array(x0_input, dtype=float).flatten()
    
    md = []
    md.append("## ❖ PHƯƠNG PHÁP LẶP JACOBI (Giải hệ $Ax = b$)")
    md.append("---\n")
    
    md.append("### I. CHÉO TRỘI VÀ BIẾN ĐỔI")
    md.append(f"$$ A = {matrix_to_latex(A)} $$")
    
    # Kiểm tra chéo trội hàng
    is_diagonally_dominant = True
    for i in range(n):
        if np.abs(A[i, i]) <= np.sum(np.abs(A[i])) - np.abs(A[i, i]):
            is_diagonally_dominant = False
            break
            
    if is_diagonally_dominant:
        md.append("- **Đánh giá:** Ma trận A là ma trận **chéo trội hàng ngặt**. Thuật toán Jacobi **chắc chắn hội tụ**!")
    else:
        md.append("- **Cảnh báo:** Ma trận A **KHÔNG** chéo trội hàng ngặt. Thuật toán có thể phân kỳ.")

    # Biến đổi
    D = np.diag(np.diag(A))
    L_plus_U = A - D
    D_inv = np.linalg.inv(D)
    B = -D_inv @ L_plus_U
    d = D_inv @ b
    
    md.append("Chuyển hệ phương trình về dạng $x = Bx + d$ với $B = -D^{-1}(L+U)$ và $d = D^{-1}b$:")
    md.append(f"$$ B = {matrix_to_latex(B)} $$")
    md.append(f"$$ d = {matrix_to_latex(d)} $$\n")
    
    q = np.linalg.norm(B, np.inf)
    md.append(f"- **Chuẩn vô cùng của B:** $q = ||B||_\\infty = {q:.5f}$")
    
    md.append("\n---\n### II. BẢNG QUÁ TRÌNH LẶP")
    
    table = ["| $k$ | " + " | ".join([f"$x_{i+1}$" for i in range(n)]) + " | Sai số $||x^{(k)} - x^{(k-1)}||_\\infty$ |",
             "|---|" + "|".join(["---"] * n) + "|---|"]
             
    history = [x_k.copy()]
    k = 0
    while True:
        if k == 0:
            row_str = " | ".join([f"{v:.5f}" for v in x_k])
            table.append(f"| {k} | {row_str} | - |")
        else:
            diff = np.linalg.norm(history[k] - history[k-1], np.inf)
            row_str = " | ".join([f"{v:.5f}" for v in x_k])
            table.append(f"| {k} | {row_str} | {diff:.5f} |")
            
            stop_eps = (epsilon is not None) and (diff <= epsilon or (q/(1-q))*diff <= epsilon if q < 1 else False)
            stop_iter = (num_iters is not None) and (k >= num_iters)
            if stop_eps or stop_iter:
                break
                
        # Lặp Jacobi thực chất là lặp đơn trên dạng x = Bx+d
        x_next = B @ x_k + d
        history.append(x_next.copy())
        x_k = x_next
        k += 1
        
    md.extend(table)
    
    md.append("\n---\n### III. KẾT LUẬN")
    md.append(f"Hệ dừng lại tại bước lặp $k = {k}$. Nghiệm gần đúng là:")
    md.append(f"$$ X^{{({k})}} \\approx {matrix_to_latex(x_k, precision=5)} $$")
    
    diff_final = np.linalg.norm(history[-1] - history[-2], np.inf)
    if q < 1:
        sai_so_hau_nghiem = (q / (1 - q)) * diff_final
        md.append(f"**Sai số hậu nghiệm:** $\\frac{{q}}{{1-q}} ||x^{{({k})}} - x^{{({k-1})}}||_\\infty = {sai_so_hau_nghiem:.5e}$")
    else:
        md.append(f"**Sai số thực tế (khoảng cách 2 bước cuối):** $||x^{{({k})}} - x^{{({k-1})}}||_\\infty = {diff_final:.5e}$")
        
    display(Markdown('\n'.join(md)))

# Ma trận cung cầu A (Kích thước 10x10)

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [10.0, 1.0,  2.0],
    [ 1.0, 10.0, 3.0],
    [ 2.0,  3.0, 10.0]
], dtype=float)

b = np.array([7.0, 8.0, 9.0], dtype=float)
# ═══════════════════════════════════════════════════════════════════

Jacobi_Ax_b(A, b, num_iters=100, epsilon=1e-6)
